# Tutorial 1: Rust (1987) Bus Engine Replacement

This tutorial replicates the estimation from John Rust's seminal 1987 Econometrica paper,
"Optimal Replacement of GMC Bus Engines: An Empirical Model of Harold Zurcher."

## The Economic Problem

Harold Zurcher is a superintendent at the Madison Metropolitan Bus Company. Each month,
he must decide whether to **keep** running each bus's current engine or **replace** it.

This is a classic **dynamic discrete choice** problem:
- **Dynamic**: Today's decision affects tomorrow's state (mileage accumulates)
- **Discrete**: Binary choice (keep or replace)
- **Stochastic**: Future mileage is uncertain

### The Model

**State space**: Mileage discretized into 90 bins (each ~5,000 miles)

**Actions**:
- Keep (a=0): Continue with current engine
- Replace (a=1): Install new engine, reset mileage to 0

**Flow utilities**:
$$u(s, \text{keep}) = -\theta_c \cdot \text{mileage}(s) + \varepsilon_0$$
$$u(s, \text{replace}) = -RC + \varepsilon_1$$

where:
- $\theta_c$ = operating cost (maintenance increases with mileage)
- $RC$ = replacement cost (fixed cost of new engine)
- $\varepsilon$ = i.i.d. Type I Extreme Value shocks

**Goal**: Estimate $(\theta_c, RC)$ from observed replacement decisions.

### Reference

Rust, J. (1987). "Optimal Replacement of GMC Bus Engines: An Empirical Model of Harold Zurcher."
*Econometrica*, 55(5), 999-1033.

In [ ]:
# Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# econirl imports
from econirl import RustBusEnvironment, LinearUtility, NFXPEstimator
from econirl.datasets import load_rust_bus
from econirl.simulation import simulate_panel

print("econirl loaded successfully!")

## Step 1: Load the Data

We'll use the built-in Rust bus dataset, which contains monthly observations of
bus mileage and replacement decisions.

In [ ]:
# Load the dataset
df = load_rust_bus()

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nNumber of buses: {df['bus_id'].nunique()}")
print(f"Number of groups: {df['group'].nunique()}")
print(f"\nFirst few rows:")
df.head(10)

In [ ]:
# Summary statistics
print("Summary Statistics")
print("=" * 50)
print(f"Total observations: {len(df):,}")
print(f"Overall replacement rate: {df['replaced'].mean():.2%}")
print(f"Mean mileage bin: {df['mileage_bin'].mean():.1f}")
print(f"Max mileage bin: {df['mileage_bin'].max()}")

print("\nReplacements by group:")
print(df.groupby('group')['replaced'].agg(['count', 'sum', 'mean']).round(3))

In [ ]:
# Visualize the data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mileage distribution
axes[0].hist(df['mileage_bin'], bins=50, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].set_xlabel('Mileage Bin')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Mileage States')

# Empirical replacement rate by mileage
replace_by_mileage = df.groupby('mileage_bin')['replaced'].agg(['mean', 'count'])
valid = replace_by_mileage['count'] >= 10
axes[1].scatter(
    replace_by_mileage.index[valid], 
    replace_by_mileage['mean'][valid],
    s=replace_by_mileage['count'][valid] / 5,
    alpha=0.6,
    color='coral'
)
axes[1].set_xlabel('Mileage Bin')
axes[1].set_ylabel('Replacement Rate')
axes[1].set_title('Empirical Replacement Rate by Mileage\n(bubble size = # observations)')

plt.tight_layout()
plt.show()

## Step 2: Set Up the Environment

The `RustBusEnvironment` encodes the structural assumptions of Rust's model:
- State transitions (how mileage evolves)
- Action space (keep vs replace)
- Discount factor and scale parameter

In [ ]:
# Create the environment
# We don't set true parameters since we're estimating from data
env = RustBusEnvironment(
    num_mileage_bins=90,
    discount_factor=0.9999,
    scale_parameter=1.0,
)

print(env.describe())

## Step 3: Convert Data to Panel Format

econirl's estimators expect data in `Panel` format - a collection of trajectories
where each trajectory is a sequence of (state, action) pairs for one individual.

In [ ]:
# Load data directly as Panel
panel = load_rust_bus(as_panel=True)

print(f"Panel Summary:")
print(f"  Individuals: {panel.num_individuals}")
print(f"  Total observations: {panel.num_observations:,}")

## Step 4: Set Up NFXP Estimation

The **Nested Fixed Point (NFXP)** algorithm is Rust's original estimation method:

1. **Outer loop**: Search over parameters $(\theta_c, RC)$
2. **Inner loop**: For each parameter guess, solve the dynamic programming problem
   to get choice probabilities
3. **Objective**: Maximize the log-likelihood of observed choices

The key insight is that given the logit error structure, choice probabilities have
a closed form once we know the value function.

In [ ]:
# Create utility specification
utility = LinearUtility.from_environment(env)

print(f"Utility Specification:")
print(f"  Parameters to estimate: {utility.parameter_names}")
print(f"  Feature matrix shape: {utility.feature_matrix.shape}")

In [ ]:
# Create the NFXP estimator
estimator = NFXPEstimator(
    se_method="asymptotic",
    optimizer="L-BFGS-B",
    verbose=True,
)

print("NFXP Estimator ready.")

## Step 5: Run Estimation

In [ ]:
# Run estimation
result = estimator.estimate(
    panel=panel,
    utility=utility,
    problem=env.problem_spec,
    transitions=env.transition_matrices,
)

print("\nEstimation complete!")

## Step 6: View Results

econirl provides StatsModels-style output with parameter estimates,
standard errors, t-statistics, p-values, and confidence intervals.

In [ ]:
# Print the summary table
print(result.summary())

In [ ]:
# Access individual results
print("Detailed Results")
print("=" * 50)

for i, name in enumerate(result.parameter_names):
    est = result.parameters[i].item()
    se = result.standard_errors[i].item()
    ci_low, ci_high = result.confidence_interval(alpha=0.05)
    
    print(f"\n{name}:")
    print(f"  Estimate: {est:.6f}")
    print(f"  Std Error: {se:.6f}")
    print(f"  95% CI: [{ci_low[i].item():.6f}, {ci_high[i].item():.6f}]")

## Step 7: Model Fit

Let's compare the model's predicted choice probabilities with the empirical data.

In [ ]:
# Get predicted policy
predicted_policy = result.policy.numpy()

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

# Model predictions
states = np.arange(env.num_states)
ax.plot(states, predicted_policy[:, 1], 'b-', linewidth=2, 
        label='Model prediction', zorder=2)

# Empirical rates
ax.scatter(
    replace_by_mileage.index[valid], 
    replace_by_mileage['mean'][valid],
    s=np.sqrt(replace_by_mileage['count'][valid]) * 3,
    alpha=0.5,
    color='coral',
    label='Empirical (size = sqrt(n))',
    zorder=1
)

ax.set_xlabel('Mileage Bin')
ax.set_ylabel('P(Replace)')
ax.set_title('Model Fit: Predicted vs Empirical Replacement Probability')
ax.legend()
ax.set_xlim(0, 70)

plt.tight_layout()
plt.show()

# Goodness of fit statistics
gof = result.goodness_of_fit
print(f"\nGoodness of Fit:")
print(f"  Log-likelihood: {result.log_likelihood:,.2f}")
print(f"  AIC: {gof.aic:,.2f}")
print(f"  BIC: {gof.bic:,.2f}")
if gof.prediction_accuracy is not None:
    print(f"  Prediction Accuracy: {gof.prediction_accuracy:.1%}")

## Step 8: Export Results

econirl supports exporting results to various formats for papers and reports.

In [ ]:
# Export to DataFrame
results_df = result.to_dataframe()
print("Results as DataFrame:")
results_df

In [ ]:
# Export to LaTeX
latex_table = result.to_latex(caption="Rust (1987) Replication Results")
print("LaTeX Table:")
print(latex_table)

## Summary

In this tutorial, we:

1. **Loaded real data** using `load_rust_bus()`
2. **Set up the model** with `RustBusEnvironment` and `LinearUtility`
3. **Estimated parameters** using NFXP
4. **Examined results** with StatsModels-style output
5. **Assessed model fit** by comparing predictions to data
6. **Exported results** for publication

The estimated parameters can be interpreted as:
- **Operating cost ($\theta_c$)**: The disutility per mileage bin from maintenance costs
- **Replacement cost ($RC$)**: The fixed disutility from replacing an engine

### Next Steps

- See [Tutorial 2](simulated_data_workflow.ipynb) for Monte Carlo analysis of parameter recovery
- See [Tutorial 3](real_data_example.ipynb) for applying econirl to other datasets